# steering-resistance — Colab launcher

Thin launcher: clones the repo, installs the `steer` package, runs the tested pipeline. **No science lives in this notebook** — all logic is in the package, so a Colab run and a local run are identical.

**Before running:**
1. Set a GPU runtime: Runtime → Change runtime type → GPU (T4 is fine for the 0.5B smoke config; 7B needs A100/L4).
2. Add Colab secrets (🔑 in the left sidebar, toggle *Notebook access*):
   - `GH_TOKEN` — **required while the repo is private** (a GitHub token with read access to the repo).
   - `HF_TOKEN`, `WANDB_API_KEY` — optional; only needed to push the adapter / log to W&B on real runs. The smoke run needs neither.

In [ ]:
!nvidia-smi -L    # confirm a GPU is attached

In [ ]:
# Clone (or update) the repo and install the package. Uses GH_TOKEN for the
# private repo; falls back to an anonymous clone if the repo is public.
# Colab already ships a CUDA torch, so pip only adds/upgrades the rest.
import os, subprocess
from google.colab import userdata
try:
    _tok = userdata.get("GH_TOKEN")
    REPO = f"https://{_tok}@github.com/JacoDuToit11/steering-resistance.git"
except Exception:
    REPO = "https://github.com/JacoDuToit11/steering-resistance.git"  # public fallback
if not os.path.isdir("steering-resistance"):
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
%cd steering-resistance
!git pull -q
!pip install -q -e ".[dev,track]"

In [ ]:
# Auth for tracking/backup (optional — skip for the smoke run). Missing token =
# that tracker stays off; the pipeline is fail-soft and never depends on it.
for key in ("HF_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[key] = userdata.get(key)
        print(f"{key}: set")
    except Exception:
        print(f"{key}: not found (that tracker will stay off)")

In [ ]:
# Machinery check (fast) — the three gradient-flow asserts on the 0.5B config.
!python scripts/verify_hooks.py configs/smoke_0.5b.yaml

In [ ]:
# Run. Defaults to the 0.5B smoke config (free T4, ~5 min, no tokens needed).
# For the real 7B run: CONFIG="configs/qwen7b_popqa.yaml", set HF_USER +
# WANDB_PROJECT, and use an A100/L4 runtime.
import yaml
CONFIG = "configs/smoke_0.5b.yaml"
HF_USER = None            # e.g. "JacoDuToit11" → pushes to <user>/steer-<run>
WANDB_PROJECT = None      # e.g. "steering-resistance"

cfg = yaml.safe_load(open(CONFIG))
run = os.path.splitext(os.path.basename(CONFIG))[0]
cfg["hub_repo_id"] = f"{HF_USER}/steer-{run}" if HF_USER else None
cfg["wandb_project"] = WANDB_PROJECT
yaml.safe_dump(cfg, open(CONFIG, "w"), sort_keys=False)
print("config:", CONFIG, "| hub_repo_id:", cfg["hub_repo_id"], "| wandb:", cfg["wandb_project"])

!python scripts/run.py {CONFIG}